# Program - 

**Purpose**

**Data**

ncdata in SCM, T3:/home/j07hsu00/taiesm/inputdata/atm/cam/inic/gaus/cami_0000-01-01_64x128_L30_c090102.nc 

TaiESM1 cami_mam3, T3: /home/j07hsu00/taiesm/inputdata/atm/cam/inic/fv/cami-mam3_0000-01-01_0.9x1.25_L30_c100618.nc

**Content**
- read aerosol fields from cami_mam3 and then save into a new ncdata file

**Author:** Yi-Hsuan Chen (yihsuan@umich.edu)

**Date:** 

**Reference program:**



# Add aerosol fields into SCM ncdata & write out

In [14]:
import xarray as xr
import xesmf as xe
import numpy as np
from datetime import datetime

# === User inputs ===
file_in = 'cami-mam3_0000-01-01_0.9x1.25_L30_c100618.nc'      # Source file
file_out = 'cami_0000-01-01_64x128_L30_c090102.nc'             # Target file
var_list = [
    "H2O2", "H2SO4", "DMS", "SOAG",
    "so4_a1", "pom_a1", "soa_a1", "bc_a1", "dst_a1", "ncl_a1", "num_a1",
    "so4_a2", "soa_a2", "ncl_a2", "num_a2",
    "SO2", "dst_a3", "ncl_a3", "num_a3", "pom_a1", "so4_a3", 
]

#var_list = ["soa_a2"]

file_write_out = "./cami_0000-01-01_64x128_L30_c090102.yhc0701_aeroIC_cami-mam3.nc"

#do_write_out = False
do_write_out = True
#do_check = False
do_check = True

# === Load datasets ===
ds_in = xr.open_dataset(file_in)
ds_out = xr.open_dataset(file_out)
#print(ds_out["T"].values)

# Clip latitudes slightly to avoid edge issues (if needed)
ds_in['lat'] = ds_in['lat'].clip(min=-89.9999, max=89.9999)
ds_out['lat'] = ds_out['lat'].clip(min=-89.9999, max=89.9999)

# Create regridder (based on lat/lon grid only)
# Assumes ds_in and ds_out have 'lat' and 'lon' coordinates
regridder = xe.Regridder(
    ds_in, ds_out,
    method='bilinear',
    extrap_method='nearest_s2d',
    periodic=True
)

# === Interpolate variables with looping over time and lev ===
for var in var_list:
    if var in ds_in:
        print(f"Interpolating variable: {var}")

        data_in = ds_in[var]
        # Get dimension sizes
        time_len = data_in.sizes['time']
        lev_len = data_in.sizes['lev']
        lat_len_out = ds_out.sizes['lat']
        lon_len_out = ds_out.sizes['lon']

        # Prepare empty numpy array to hold interpolated data
        interp_data = np.empty((time_len, lev_len, lat_len_out, lon_len_out), dtype=data_in.dtype)

        # Loop through time and level
        for t in range(time_len):
            for l in range(lev_len):
                slice_2d = data_in.isel(time=t, lev=l)
                interp_slice = regridder(slice_2d)
                #print(np.isnan(interp_slice).any().values)  # Should be False ideally
                interp_data[t, l, :, :] = interp_slice.values
                #interp_data[t, l, :, :] = 1e-13

        # Build DataArray with correct dims and coords
        #print(np.isnan(interp_data).any())  # Should be False ideally
        #print(interp_data)

        interp_var = xr.DataArray(
            interp_data.astype('float64'),
            dims=['time', 'lev', 'lat', 'lon'],
            coords={
                'time': ds_out['time'],
                'lev': ds_out['lev'],
                'lat': ds_out['lat'],
                'lon': ds_out['lon']
            }
        )
        
        # Assign interpolated variable to ds_out
        interp_var.attrs = data_in.attrs  # Preserve metadata
        ds_out[var] = interp_var

    else:
        print(f"⚠️ Variable '{var}' not found in input dataset. Skipping.")

# Optional: check some points before saving
#do_check = True
if (do_check):
    #points_to_check = [(0.0, 120.0), (10.0, 130.0), (-20.0, 150.0)]
    points_to_check = [(31.5,238.5)]
    for var in var_list:
        if var in ds_in and var in ds_out:
            print(f"\nChecking variable: {var}")
            for lat_val, lon_val in points_to_check:
                val_in = ds_in[var].sel(lat=lat_val, lon=lon_val, method='nearest')
                val_out = ds_out[var].sel(lat=lat_val, lon=lon_val, method='nearest')
                print("----------")
                print(f" At (lat={lat_val}, lon={lon_val}):")
                print(f"   Input : {val_in.values}")
                print("")
                print(f"   Output: {val_out.values}")

# Save updated output dataset (optional)
#do_write_out = False
if do_write_out:
    current_date = datetime.now().isoformat()

    ds_out.attrs.update({
        'Modification': 'add aerosol fields into SCM ncdata',
        'SCM_original_ncdata': '/taiesm/inputdata/atm/cam/inic/gaus/cami_0000-01-01_64x128_L30_c090102.nc',
        'file_aerosol_fields': '/taiesm/inputdata/atm/cam/inic/fv/cami-mam3_0000-01-01_0.9x1.25_L30_c100618.nc',
        'history': f'Created on {current_date}',
        'contact': 'Yi-Hsuan Chen (yihsuanc@as.edu.tw)'
    })
    
    #print(ds_out.attrs)
    ds_out.to_netcdf(file_write_out)
    print(f"✅ NetCDF file written: {file_write_out}")

#ds_out["T"].values

Interpolating variable: H2O2
Interpolating variable: H2SO4
Interpolating variable: DMS
Interpolating variable: SOAG
Interpolating variable: so4_a1
Interpolating variable: pom_a1
Interpolating variable: soa_a1
Interpolating variable: bc_a1
Interpolating variable: dst_a1
Interpolating variable: ncl_a1
Interpolating variable: num_a1
Interpolating variable: so4_a2
Interpolating variable: soa_a2
Interpolating variable: ncl_a2
Interpolating variable: num_a2
Interpolating variable: SO2
Interpolating variable: dst_a3
Interpolating variable: ncl_a3
Interpolating variable: num_a3
Interpolating variable: pom_a1
Interpolating variable: so4_a3

Checking variable: H2O2
----------
 At (lat=31.5, lon=238.5):
   Input : [[1.4529591e-12 3.1856187e-12 2.3225438e-12 1.6241418e-12 1.0336824e-12
  6.3765150e-13 2.6834643e-13 2.4491702e-13 4.5350846e-13 2.1185493e-12
  4.2393823e-12 8.1741627e-12 1.4877263e-11 2.4962770e-11 4.6118588e-11
  8.1203225e-11 1.3562855e-10 1.8691945e-10 2.4180222e-10 2.7497535e-10